# Expense Analyst Agent — Walkthrough

Run **top to bottom**. Need a `.env` with `AZURE_OPENAI_ENDPOINT` and `AZURE_OPENAI_API_KEY`.

1. **Install + imports** — packages, then SDK / PDF / Gradio imports.
2. **Azure setup** — load keys, disable tracing, wire the chat model.
3. **Tools** — PDF text, CSV parse/validate, spending summary by category.
4. **Agent** — `ExpenseAnalystAgent` uses those tools.
5. **Tests** — one PDF run, one CSV run. Check the printed summaries.
6. **Chat** — last code cell opens Gradio. Sample files: `data/MonthlySpendingData_Placeholder.pdf`, `data/sample_transactions.csv`.

## 1. Install packages

Run this once. Installs the Agents SDK, Azure OpenAI client, PDF reader, Gradio, and pandas.

In [4]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio pandas

Resolved 78 packages in 364ms
Checked 78 packages in 9ms


## Imports

Brings in dotenv, Azure OpenAI, Agents SDK, PDF reader, Gradio, and pandas. No output expected.

In [5]:
from dotenv import load_dotenv
import os

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr
import pandas as pd

c:\Users\Danie\Downloads\Final Project\FinalProject_IT3025C\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Azure keys

Reads `.env`. Both prints should be `True`. If not, stop and fix the env file.

In [6]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Wire the Azure model

Turns tracing off, then builds `azure_model` for every agent call below.

In [7]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. PDF tool

Pulls text from `data/MonthlySpendingData_Placeholder.pdf`. `extract_pdf_text` is the tool the agent can call. You should see a character-count print.

In [8]:
def read_pdf(file_path: str) -> str:
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# AI Agent Tool
@function_tool
def extract_pdf_text(file_path: str) -> str:
    """Extract text content from a PDF file."""
    return read_pdf(file_path)


# PDF Collection
MONTHLY_EXPENSES_COLLECTION = {
    "monthly_expenses_pdf_1": {
        "filename": "MonthlySpendingData_Placeholder.pdf",
        "content": read_pdf(
            "data/MonthlySpendingData_Placeholder.pdf"
        )
    }
}


spending_data_text = MONTHLY_EXPENSES_COLLECTION[
    "monthly_expenses_pdf_1"
]["content"]

print("Loaded PDF text length:", len(spending_data_text), "characters")

Loaded PDF text length: 275 characters


## 5. CSV validation

Banks export CSVs differently — this file is a placeholder. Required columns: `date`, `description`, `amount`. Bad rows raise an error.

In [9]:
# Validate csv file
def load_validated_csv(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)

    required = {"date", "description", "amount"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    if df[["date", "amount", "description"]].isna().any().any():
        raise ValueError("CSV contains invalid or missing values")

    if df["description"].astype(str).str.strip().eq("").any():
        raise ValueError("CSV contains empty descriptions")

    return df

## 6. CSV parse tool

Wraps the validator and returns transactions as JSON. On a bad file it returns `success: False` plus the error.

In [10]:
@function_tool
def parse_csv_transactions_tool(file_path: str) -> dict:
    """Parse and validate transactions from a CSV file."""
    try:
        df = load_validated_csv(file_path)

        return {
            "success": True,
            "transaction_count": len(df),
            "transactions": df.assign(
                date=df["date"].dt.strftime("%Y-%m-%d")
            ).to_dict(orient="records")
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 7. Spending summary tool

Totals spend and buckets merchants into Groceries / Dining / Transportation / Other.

In [11]:
@function_tool
def calculate_spending_summary_tool(file_path: str) -> dict:
    """Calculate total spending and expenses by category from a CSV file."""
    try:
        df = load_validated_csv(file_path)

        # Categorize transactions based on merchant descriptions
        def categorize(description: str) -> str:
            description = description.lower()

            if any(word in description for word in ["walmart", "kroger", "aldi"]):
                return "Groceries"
            elif any(word in description for word in ["starbucks", "mcdonald", "chipotle"]):
                return "Dining"
            elif any(word in description for word in ["shell", "exxon", "uber"]):
                return "Transportation"
            else:
                return "Other"

        df["category"] = df["description"].apply(categorize)

        spending_by_category = (
            df.groupby("category")["amount"]
            .sum()
            .round(2)
            .to_dict()
        )

        return {
            "success": True,
            "total_spending": round(float(df["amount"].sum()), 2),
            "transaction_count": len(df),
            "spending_by_category": spending_by_category
        }

    except (
        OSError,
        ValueError,
        pd.errors.ParserError,
        UnicodeError
    ) as e:
        return {
            "success": False,
            "error": str(e)
        }

## 8. Save tool (stub)

Placeholder only. Not attached to the agent yet — skip mentally, do not expect it to persist data.

In [12]:
@function_tool
def save_transactions_tool():
    return

## 9. Build ExpenseAnalystAgent

System prompt + three tools: PDF extract, CSV parse, spending summary. Print should say `Agent built: ExpenseAnalystAgent`.

In [13]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Expense Analyst Agent, You are a Financial Advizing Agent that Analyzes financial data provided by users through 
CSV files, chat inputs, PDF files. You categorizes transactions, identifies spending patterns, and summarizes expenses for the user


You are to, when prompted by the user, to list the monthly expenses provided in the pdf from the MONTHLY_EXPENSES_COLLECTION and tell them how they could better optimize their spending.

. 

"""

ExpenseAnalystAgent = Agent(
    name="ExpenseAnalystAgent",
    handoff_description="Analyzes budget and summarizes spending.",
    instructions=instructions,
    tools=[extract_pdf_text, parse_csv_transactions_tool, calculate_spending_summary_tool],
    model=azure_model,
)

print("Agent built:", ExpenseAnalystAgent.name)

#when executed with tools that can read the MONTHLY_EXPENSES_COLLECTION, the chatbot should be able to properly assess the file
#for now, its a pdf as a placeholder but we can change that later. 
# We will add more to the description of the agent we are creating the tools - Chris


Agent built: ExpenseAnalystAgent


## Agent 2: Build Budget Optimizer
System prompt + three tools: PDF extract, CSV parse, Budget Plan. Print should say `Agent built: Budget Optimizer`.

In [14]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Budget Optimizer Agent, You are a Financial Advizing Agent 
that builds a Optimizer plan for the user after looking over the data from Expense Analyst Agent.


You are to, when prompted by the user, to layout a plan for them step by step on what money should be mostly spent on and what to remove for not useful spending.

. 

"""

BudgetOptimizerAgent = Agent(
    name="BudgetOptimizerAgent",
    handoff_description="Create and print a layout plan for spending.",
    instructions=instructions,
    tools=[extract_pdf_text, parse_csv_transactions_tool, calculate_spending_summary_tool],
    model=azure_model,
)

print("Agent built:", BudgetOptimizerAgent.name)


Agent built: BudgetOptimizerAgent


## 10. Test — PDF

Asks the agent to read the placeholder PDF and summarize expenses. Expect a list of bills plus optimization tips.

In [15]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/MonthlySpendingData_Placeholder.pdf and summarize my expenses."
)

print(result.final_output)

Here’s a summary of your monthly expenses from the PDF:

### Monthly expenses
- Rent: $1,000
- Gas: $200
- Groceries: $400
- Spotify: $10
- Netflix: $15
- Take-out: $50
- Car Payment: $160
- Car Insurance: $100
- Medical Insurance: $300
- Amazon orders: $100
- Miscellaneous: $400

### Total monthly spending
- **$3,500**

### Additional income note
- Money taken home: **$765**

## Quick spending observations
Your biggest expense categories are:
1. **Rent — $1,000**
2. **Groceries — $400**
3. **Miscellaneous — $400**
4. **Medical Insurance — $300**
5. **Gas — $200**

## Ways to optimize your spending
- **Review miscellaneous spending ($400):** This is one of the largest categories and may include reducible expenses.
- **Cut back on Amazon orders ($100):** Setting a monthly cap could help.
- **Reduce groceries ($400):** Meal planning, buying store brands, and shopping with a list may lower costs.
- **Limit take-out ($50):** Already modest, but reducing it further can help.
- **Evaluate su

## 11. Test — CSV

Asks the agent to read `data/sample_transactions.csv`. Expect a transaction count, total, and category breakdown.

In [16]:
result = await Runner.run(
    ExpenseAnalystAgent,
    "Read data/sample_transactions.csv")

print(result.final_output)

I read `data/sample_transactions.csv`.

Summary:
- Total spending: $1,642.54
- Number of transactions: 25

By category:
- Other: $1,267.43
- Groceries: $210.86
- Transportation: $112.75
- Dining: $51.50

Quick insight:
- Most of your spending is in Other, which is by far the largest category.
- Groceries are your next biggest expense.
- Dining is relatively low.

Optimization ideas:
- Review the Other category first, since that’s where most of your money is going. It may contain recurring bills, shopping, or uncategorized purchases that could be reduced.
- If you want tighter control, recategorizing the Other transactions would help identify specific savings opportunities.
- Groceries are moderate, so using meal planning or store budgeting could help reduce them further.
- Transportation looks manageable, but you could still watch for ride-share or fuel costs if applicable.

If you want, I can also:
- parse and list the individual transactions from the CSV, or
- give you a more detaile

## 12. Gradio chat

Opens a chat UI in the browser. Session memory goes to `user_expenses.db`. Try: *Read data/sample_transactions.csv* or *summarize my PDF expenses*.

In [17]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")

async def chat(message, history):
    result = await Runner.run(ExpenseAnalystAgent, message, session=chat_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Notes (from Chris)

Skeleton agent: one PDF for now, tools first, memory later. Last cell is teammate comments only.

In [18]:
# Hey guys, so this is basically the skeleton of the Expenses Agent, I formatted it in a way that it will read a single pdf file when given the tools to do so
#We will end up adding memory as well, preferably after the tools are implemented for this agent
#It may be a little thrown together in terms of the structure, but I made what I could on my own. If you guys have issues with it, we will modify it together. 
#-Chris